Using intuition:
Grammar Design: Write a CFG/EBNF that models your expression’s structure.

Using Lark:

1.   Tokenization: Define tokens using regex and set up a lexer.
2.  Parser Implementation: Use a parser generator to build a parser with your grammar.

4.  AST Construction: Transform the parse tree into an AST if necessary.
5.  Handling various types of inputs (newline or single line)

6. Generating all possible paths

7. Creating a graph for the paths

In [26]:
!pip install graphviz
!pip install lark
from graphviz import Digraph
from lark import Lark

In [27]:
# Create a parser to build up a syntax tree of the tokens created by the lexer
# Tree tells you what operations need to be performed and the order of operations
# Figure out tokens and match language grammar
# Figure out what makes sense and what doesn't

#/content/drive/MyDrive/Lab Work/Results_for_PNNL_March_2025/Graph_Generation/grammar.txt

## BUILDING A TREE
## NODES

file_path = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/new_updated_grammar.txt"
with open(file_path, "r") as f:
    grammar = f.read()
print(grammar)


parser = Lark(grammar, parser='lalr', start='file')


def tree_to_graphviz(tree, graph=None, parent_id=None, node_id=0):
    if graph is None:
        graph = Digraph()
    # Create a unique id for this node
    current_id = f"node{node_id}"
    # Label: use the rule name for non-terminals, token value for terminals
    label = tree.data if hasattr(tree, 'data') else str(tree)
    graph.node(current_id, label)

    # If there is a parent, create an edge
    if parent_id is not None:
        graph.edge(parent_id, current_id)

    node_id += 1

    # If it's a Tree node, iterate over children
    if hasattr(tree, 'children'):
        for child in tree.children:
            graph, node_id = tree_to_graphviz(child, graph, current_id, node_id)

    return graph, node_id



?file: expr (NEWLINE expr)*
expr: top

# OUTSIDE parens: spaces split steps
top: step (SPACE step)*
step: choice | chain

# Steps can contain choices which are comma seperated alternatives "chains"
choice: chain (COMMA chain)+ 
# Chains can contain factors which are seperated by plus or optional nodes 
chain: factor (PLUS factor | MINUS factor)*  

# factors are units such as modules or parenthesized groups
factor: unit | group
# Parenthesized group switches to "inside parens" rules
group: LPAREN inner RPAREN

# INSIDE parens: commas = alternatives, spaces = sequence
inner: seq (COMMA seq)*             
seq: chain (SPACE chain)*           

# terminals
unit: ko_id | module | start_node | sink_node

ko_id: /-?K[0-9]{5}(?:_[0-9]+)?/
module: /M[0-9]{5}/

start_node: "START"
sink_node: "SINK"

SPACE: /[ \t]+/
COMMA: ","
PLUS: "+"
MINUS: "-"
LPAREN: "("
RPAREN: ")"
NEWLINE: "\n"



In [28]:
def get_next_token_number():
    global global_token_counter
    global_token_counter += 1
    return global_token_counter

In [29]:
## LARK transformer to AST
## For processing matched rules and tokens

from lark import Transformer

class ASTNode:
    pass

class ProgramNode(ASTNode):
    def __init__(self, expressions):
        self.expressions = expressions
    def __repr__(self):
        return f"Program({self.expressions})"

class SequenceNode(ASTNode):
    def __init__(self, parts):
        self.parts = parts
    def __repr__(self):
        return f"Sequence({self.parts})"

class AlternativeNode(ASTNode):
    def __init__(self, alternatives):
        self.alternatives = alternatives
    def __repr__(self):
        return f"Alternative({self.alternatives})"

class MultiplyNode(ASTNode):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op  # either '+' or '-'
        self.right = right
    def __repr__(self):
        return f"Multiply({self.left} {self.op} {self.right})"

class StartNode(ASTNode):
    def __repr__(self):
        return "START"

class SinkNode(ASTNode):
    def __repr__(self):
        return "SINK"

class KoIDNode(ASTNode):
    def __init__(self, kid):
        self.kid = kid
    def __repr__(self):
        return f"{self.kid}"

class OptionalNode(ASTNode):
    def __init__(self, factor):
        self.factor = factor
    def __repr__(self):
        return f"Optional({self.factor})"

class ModuleNode(ASTNode):
    def __init__(self, module_id):
        self.module_id = module_id
    def __repr__(self):
        return f"Module({self.module_id})"



from lark import Transformer, Token


def _mk_sequence(items):
  parts = [x for x in items
    if not (isinstance(x, Token) and x.type == "SPACE")
    and not (isinstance(x, str) and not x.strip())]
  return parts[0] if len(parts) == 1 else SequenceNode(parts)


class ASTTransformer(Transformer):
    def file(self, items):
      return ProgramNode([x for x in items if not (isinstance(x, Token) and x.type == "NEWLINE")])

    def expr(self, items):
        # items = [START, top, SINK] (by grammar)
        return SequenceNode(items)

    # Outside-paren structure
    def top(self, items):
        # top: step (SPACE step)*  -> make a sequence of steps
        return _mk_sequence(items)

    def factor(self, items):
      return items[0]

    def unit(self, items):
      return items[0]  # lets ko_id/module handlers produce your nodes


    def step(self, items):
        # step: choice | chain
        return items[0]

    # Alternatives at depth-0 (outside parens)
    def choice(self, items):
        # choice: chain (COMMA chain)+
        branches = [x for x in items if not (isinstance(x, Token) and x.type == "COMMA")]
        return AlternativeNode(branches)


    def alt_list(self, items):
    # items may include COMMA tokens; keep only real branches
      branches = [x for x in items if not isinstance(x, Token)]
      if not branches:
          raise ValueError("alt_list has no branches")
      return AlternativeNode(branches)

    # +/- product over factors
    def chain(self, items):
        # chain: factor (PLUS factor | MINUS factor)*
        left = items[0]
        i = 1
        while i < len(items):
            tok = items[i]
            if isinstance(tok, Token) and tok.type in ("PLUS", "MINUS"):
                op = '+' if tok.type == "PLUS" else '-'
                right = items[i+1]
                i += 2
            else:
                raise ValueError(f"Unexpected token in chain at index {i}: {items[i]!r}")
            # '+' concatenates; '-' means optional right (your semantics)
            if op == '+':
                left = MultiplyNode(left, '+', right)
            else:
                left = MultiplyNode(left, '-', OptionalNode(right))
        return left

    # Groups / inside-paren structure
    def group(self, items):
        # group: LPAREN inner RPAREN  -> drop punctuation, keep inner AST
        inner = [x for x in items if not isinstance(x, Token)]
        if not inner:
            raise ValueError("Empty parenthesized group")
        return inner[0]

    def inner(self, items):
        # inner: seq (COMMA seq)*  -> alternatives of seq
        branches = [x for x in items if not (isinstance(x, Token) and x.type == "COMMA")]
        return branches[0] if len(branches) == 1 else AlternativeNode(branches)

    def seq(self, items):
        # seq: chain (SPACE chain)*  -> sequence
        return _mk_sequence(items)

    # Terminals (same as before)
    def start_node(self, _):
        return StartNode()

    def sink_node(self, _):
        return SinkNode()

    def sequence(self, items):
      return _mk_sequence(items)

    def ko_id(self, items):
        token = items[0]
        value = token.value if hasattr(token, "value") else str(token)
        unique_number = get_next_token_number()
        if value.startswith('-'):
            ko_value = value[1:]
            return OptionalNode(KoIDNode(f"{ko_value}_{unique_number}"))
        else:
            return KoIDNode(f"{value}_{unique_number}")

    def module(self, items):
      token = items[0]
      return ModuleNode(token.value if isinstance(token, Token) else str(token))

In [30]:
def get_module_expression(module_id):
    global MODULE_EXPRESSION_LOOKUP
    module_expr = MODULE_EXPRESSION_LOOKUP.get(module_id)
    if not module_expr:
        raise Exception(f"Module expression for {module_id} not found in the Excel file!")
    return module_expr

In [31]:
from lark import Token

def enumerate_paths(ast_node, visited_modules=None, allow_boundaries=True):
    if visited_modules is None:
      visited_modules = set()
    # Handle raw Lark tokens.
    if isinstance(ast_node, Token):
      # If it's a punctuation token that you want to ignore, return an empty path.
      if ast_node.type in ("COMMA", "SPACE", "NEWLINE", "LPAREN", "RPAREN", "PLUS", "MINUS"):
        return [[]]
      # Otherwise, return its value.
      return [[ast_node.value]]

    # If the node is a plain string, return it (if non-empty).
    if isinstance(ast_node, str):
      if ast_node.strip() == "":
          return [[]]
      return [[ast_node.strip()]]

    # Terminal cases (custom AST nodes)
    if isinstance(ast_node, StartNode):
        return [["START"]] if allow_boundaries else [[]]

    if isinstance(ast_node, SinkNode):
        return [["SINK"]] if allow_boundaries else [[]]

    if isinstance(ast_node, KoIDNode):
      return [[f"{ast_node.kid}"]]

    if isinstance(ast_node, ModuleNode):
        m_id = ast_node.module_id
        if m_id in visited_modules:
            raise RuntimeError(f"Cyclic module reference: {m_id}")
        expr = get_module_expression(m_id)
        nested = ASTTransformer().transform(parser.parse(preprocess_input(expr)))
        visited_modules.add(m_id)
        try:
            return enumerate_paths(nested, visited_modules, allow_boundaries=False)
        finally:
            visited_modules.remove(m_id)

    # Optional: either include the node or skip it
    if isinstance(ast_node, OptionalNode):
      include_paths = enumerate_paths(ast_node.factor, visited_modules, allow_boundaries)
      skip_paths = [[]]  # Option to bypass this factor.
      return include_paths + skip_paths

    # Multiply node: represents a binary operator connecting left and right.
    if isinstance(ast_node, MultiplyNode):
      left_paths = enumerate_paths(ast_node.left, visited_modules, allow_boundaries)
      if ast_node.op == '+':
          right_paths = enumerate_paths(ast_node.right, visited_modules, allow_boundaries)
          combined = []
          for lp in left_paths:
              for rp in right_paths:
                  combined.append(lp + rp)
          return combined
      elif ast_node.op == '-':
          right = ast_node.right
          if not isinstance(right, OptionalNode):
            right = OptionalNode(right)
          right_paths = enumerate_paths(right, visited_modules, allow_boundaries)

          combined = []
          for lp in left_paths:
              for rp in right_paths:
                  combined.append(lp + rp)
          return combined

    # Sequence node: sequentially connect all parts.
    if isinstance(ast_node, SequenceNode):
        if not ast_node.parts:
            return [[]]
        paths = enumerate_paths(ast_node.parts[0], visited_modules, allow_boundaries)
        for part in ast_node.parts[1:]:
            part_paths = enumerate_paths(part, visited_modules, allow_boundaries)
            new_paths = []
            for p in paths:
                for q in part_paths:
                    new_paths.append(p + q)
            paths = new_paths
        return paths

    # Alternative node: union the paths from each alternative.
    if isinstance(ast_node, AlternativeNode):
        paths = []
        for alternative in ast_node.alternatives:
            paths.extend(enumerate_paths(alternative, visited_modules, allow_boundaries))
        return paths

    # Program node: top-level container.
    if isinstance(ast_node, ProgramNode):
        paths = []
        for expr in ast_node.expressions:
            paths.extend(enumerate_paths(expr, visited_modules, allow_boundaries))
        return paths or [[]]

    raise Exception("Unhandled AST node type: " + str(ast_node))


In [32]:
import re
_re_space_before_plus = re.compile(r'\s+\+')
_re_space_after_plus  = re.compile(r'\+(?=\s+(?:K\d{5}|M\d{5}|\())')
_re_space_after_minus = re.compile(r'-(?=\s+(?:K\d{5}|M\d{5}|\())')
_re_dangling_minus = re.compile(r'-(?=\s*(?:SINK|$))')

def validate_operator_spacing(s: str):
    s = s or ""
    m = _re_space_before_plus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Space before '+' is not allowed near: …{s[max(0,i-6):i+8]}…")
    m = _re_space_after_plus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Space after '+' is not allowed near: …{s[max(0,i-6):i+8]}…")
    m = _re_space_after_minus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Space after '-' is not allowed near: …{s[max(0,i-6):i+8]}…")

    #'-' not followed by a factor (line end or SINK)
    m = _re_dangling_minus.search(s)
    if m:
        i = m.start()
        raise ValueError(f"Dangling '-' not followed by a factor near: …{s[max(0,i-6):i+8]}…")


KO = r'K\d{5}'
MOD_OR_GROUP_START = r'(?:M\d{5}|\()'
_KO_OR_MOD = re.compile(r'^(?:K\d{5}|M\d{5})$')

def normalize_line_operators(line: str) -> str:
    s = line or ""

    # 1) KO space--space KO  → single space (treat as step break)
    s = re.sub(rf'(?<={KO})\s*--\s*(?={KO})', ' ', s)

    # Remove double dashes separating pathway to sink
    s = re.sub(r'\s--+\s', ' ', s)
    s = re.sub(r'\s*--+\s*', ' ', s)

    # 3) space BEFORE '-' is allowed; normalize it away if next is KO/M/(
    s = re.sub(rf'\s+-(?=(?:{KO}|{MOD_OR_GROUP_START}))', '-', s)

    # 4) collapse multi-spaces to one (keep sequences readable)
    s = re.sub(r'[ \t]+', ' ', s).strip()
    return s

def _split_top_level_by_commas(s: str):
    """Split on commas at paren-depth 0. Return list of trimmed items."""
    items, buf, depth = [], [], 0
    for ch in s:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth = max(0, depth - 1)
        if ch == ',' and depth == 0:
            item = ''.join(buf).strip()
            if item:
                items.append(item)
            buf = []
        else:
            buf.append(ch)
    tail = ''.join(buf).strip()
    if tail:
        items.append(tail)
    return items


def _group_is_seq(x: str) -> bool:
    """True if x == '( ... )' and its top-level inside has SPACE but no COMMA."""
    x = x.strip()
    if not (x.startswith('(') and x.endswith(')')):
        return False
    inner = x[1:-1]
    depth = 0
    has_space = False
    has_comma = False
    for ch in inner:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        elif depth == 0:
            if ch == ',':
                has_comma = True
            elif ch.isspace():
                has_space = True
    return has_space and not has_comma

def _is_simple_token(x: str) -> bool:
    """A single KO or module id, no spaces/commas/paren."""
    x = x.strip()
    return bool(_KO_OR_MOD.fullmatch(x))


def normalize_top_level_alternatives(line: str) -> str:
    return line


def preprocess_input(raw_input):
    # Convert sequences of 10 or more spaces to a newline.
    raw_input = re.sub(r' {5,}', '\n', raw_input)

    lines = [line.strip() for line in raw_input.splitlines() if line.strip() != ""]
    wrapped_lines = []
    for line in lines:
      norm = normalize_line_operators(line)
      validate_operator_spacing(norm)
      norm = normalize_top_level_alternatives(norm)
      if not norm.startswith("START"):
          norm = "START " + norm
      if not norm.endswith("SINK"):
          norm = norm + " SINK"
      wrapped_lines.append(norm)
    return "\n".join(wrapped_lines)

In [33]:
def generate_node_adjacency_list(unique_paths):
  adjacency_list = {}
  node_set = set()

  for path in unique_paths:
      for i in range(len(path)):
          node_set.add(path[i])
          if i < len(path) - 1:
              src = path[i]
              dst = path[i+1]

              if (src == "START" and dst == "START") or (src == "SINK" and dst == "SINK") or (src == "START" and dst == "SINK") or (src == "SINK" and dst == "START"):
                  continue

              if src == dst:
                  continue
              if src not in adjacency_list:
                  adjacency_list[src] = set()
              adjacency_list[src].add(dst)

  # Convert the adjacency set values to lists for printing (optional)
  for src in adjacency_list:
      adjacency_list[src] = list(adjacency_list[src])

  node_list = sorted(list(node_set))  # Sorted for consistency

  return node_list, adjacency_list

#From sink as TW suggested
def assign_node_levels(unique_paths):
    """
    Assigns levels such that:
      - START is level 1
      - SINK is the largest level
      - For any node, its level depends on its shortest distance from SINK.
        If a node appears on multiple paths, we use the minimum (shortest) distance.
    """

    # 1) Calculate the minimum distance from SINK for each node.
    sink_dist = {}
    for path in unique_paths:
        # reverse so that SINK is at position 0
        reversed_path = path[::-1]

        for dist, node in enumerate(reversed_path):
            # if node is seen for the first time or
            # the newly computed distance is smaller, update sink_dist
            if node not in sink_dist:
                sink_dist[node] = dist
            else:
                sink_dist[node] = min(sink_dist[node], dist)

    # 2) Get the maximum distance from SINK across all nodes
    if sink_dist:
        max_distance = max(sink_dist.values())
    else:
        max_distance = 0

    # 3) Convert distance-from-SINK into levels
    #    farthest from SINK => level 1
    #    closest to SINK => level (max_distance + 1)
    node_levels = {}
    for node, dist in sink_dist.items():
        node_levels[node] = (max_distance - dist) + 1

    # 4) Force START at level 1, SINK at the maximum level
    node_levels["START"] = 0
    node_levels["SINK"] = max_distance + 1

    return node_levels




In [34]:
def load_module_expressions(input_excel):
    """
    Loads module expressions from the given Excel file into a dictionary.
    Assumes the Excel has columns "Module ID" and "Module_Expression".
    """
    df = pd.read_excel(input_excel)
    module_dict = {}
    for index, row in df.iterrows():
        module_id = str(row["Module ID"]).strip()
        module_expr = str(row["Module_Expression"]).strip()
        if module_expr:  # Only add if the expression is not empty.
            module_dict[module_id] = module_expr
    return module_dict




In [35]:
def main(raw_input, output_dir, module_id):

    # Reset the global token counter for every module.
    global global_token_counter
    global_token_counter = 0

    # Preprocess input (including converting long sequences of spaces to newline).
    input_text = preprocess_input(raw_input)
    print("Preprocessed Input:")
    print(input_text)

    # Create a new parser (using your grammar)
    parser = Lark(grammar, parser='lalr', start='file')

    # Process each line separately if multiple lines.
    lines = [line.strip() for line in input_text.splitlines() if line.strip() != ""]
    all_paths = []
    if len(lines) > 1:
        for line in lines:
            print("\nProcessing line:")
            print(line)
            parse_tree = parser.parse(line)
            transformer = ASTTransformer()
            ast = transformer.transform(parse_tree)
            print("AST:")
            print(ast)
            paths = enumerate_paths(ast)
            all_paths.extend(paths)
    else:
        parse_tree = parser.parse(input_text)
        print("Parse Tree:")
        print(parse_tree.pretty())
        transformer = ASTTransformer()
        ast = transformer.transform(parse_tree)
        print("AST:")
        print(ast)
        all_paths = enumerate_paths(ast)

    # Remove duplicate paths.
    unique_paths = list({tuple(path) for path in all_paths})
    print("\nEnumerated Paths:")
    for path in unique_paths:
        print(" -> ".join(path))

    # Generate node and adjacency lists.
    node_list, adjacency_list = generate_node_adjacency_list(unique_paths)
    print("Adjacency List:")
    for node, neighbors in adjacency_list.items():
        print(f"{node}: {neighbors}")
    print("\nNode List:")
    print(node_list)

    # Compute node levels (i.e. step numbers).
    node_levels = assign_node_levels(unique_paths)
    print("Node Levels:")
    for node in sorted(node_levels):
        print(f"{node}: Level {node_levels[node]}")

    # Create a directed graph with NetworkX.
    G = nx.DiGraph()
    for node in node_list:
        G.add_node(node)
    for src, neighbors in adjacency_list.items():
        for dst in neighbors:
            G.add_edge(src, dst)

    # Create custom labels: node name and step number.
    node_labels = {node: f"{node}\nStep {node_levels[node]}" for node in G.nodes()}

    pos = nx.nx_pydot.graphviz_layout(G, prog='dot')
    plt.figure(figsize=(12,12))
    nx.draw(G, pos, with_labels=True, labels=node_labels, node_size=400, node_color='lightblue',
            arrows=True, arrowstyle='->', arrowsize=20)
    edge_labels = nx.get_edge_attributes(G, 'label')
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.title("Directed Graph of Module " + str(raw_input))
    plt.axis('off')

    png_file_path = os.path.join(output_dir, f"module_{module_id}_graph.png")
    plt.savefig(png_file_path)
    plt.close()
    print(f"Graph saved as PNG: {png_file_path}")

    # Save the adjacency list as JSON.
    adj_file_path = os.path.join(output_dir, f"module_{module_id}_adjacency.json")
    with open(adj_file_path, "w") as f:
        json.dump(adjacency_list, f, indent=4)
    print(f"Adjacency list saved to: {adj_file_path}")

    # Save the node levels (with step numbers) as JSON.
    node_list_file_path = os.path.join(output_dir, f"module_{module_id}_nodes.json")
    with open(node_list_file_path, "w") as f:
        json.dump(node_levels, f, indent=4)
    print(f"Node list (with step numbers) saved to: {node_list_file_path}")

    # Create a dictionary of paths where each key is an index (starting at 1)
    # and the value is a comma-separated string of KO indexes (tokens starting with "K").
    paths_dict = {}
    for i, path in enumerate(unique_paths, start=1):
        ko_tokens = [token for token in path if token.startswith("K")]
        paths_dict[str(i)] = ", ".join(ko_tokens)

    paths_file_path = os.path.join(output_dir, f"module_{module_id}_paths.json")
    with open(paths_file_path, "w") as f:
        json.dump(paths_dict, f, indent=4)
    print(f"Module paths saved to: {paths_file_path}")

    return adj_file_path, node_list_file_path, png_file_path, paths_file_path

In [36]:
#raw = "(K00844,K12407,K00845,K25026,K00886,K08074,K00918) (K01810,K06859,K13810,K15916) (K00850,K16370,K21071,K24182,K00918) (K01623,K01624,K11645,K16305,K16306) K01803 ((K00134,K00150) K00927,K11389) (K01834,K15633,K15634,K15635) (K01689,K27394) (K00873,K12406)"

#unique_paths, node_list, adjacency = run_once(raw)

In [37]:
import pandas as pd
global global_token_counter
global_token_counter = 0

input_excel = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/KEGG_Prokaryote_Modules_with_Definitions.xlsx"
data = pd.read_excel(input_excel)
#raw = "K09733 K19793 K07144             K18933 K06914             K07072"
print(preprocess_input(raw))   # should preserve and wrap START/SINK
row = data[data['Module ID'].astype(str).str.strip() == 'M00935'].iloc[0]
raw = str(row['Module_Expression'])
print("RAW REPR:", repr(raw))
print("AFTER normalize:", repr(normalize_line_operators(raw)))
print("AFTER preprocess:", repr(preprocess_input(raw)))
parser = Lark(grammar, parser='lalr', start='file')
tree = parser.parse(preprocess_input(raw))
ast = ASTTransformer().transform(tree)
paths = enumerate_paths(ast)
for p in {tuple(x) for x in paths}:
    print(" -> ".join(p))


START K09733 K19793 K07144 SINK
START K18933 K06914 SINK
START K07072 SINK
RAW REPR: 'K09733 K19793 K07144\nK18933 K06914\nK07072'
AFTER normalize: 'K09733 K19793 K07144\nK18933 K06914\nK07072'
AFTER preprocess: 'START K09733 K19793 K07144 SINK\nSTART K18933 K06914 SINK\nSTART K07072 SINK'
START -> K18933_4 -> K06914_5 -> SINK
START -> K07072_6 -> SINK
START -> K09733_1 -> K19793_2 -> K07144_3 -> SINK


In [38]:
import re
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from graphviz import Digraph
from lark import Lark, Transformer, Token


if __name__ == '__main__':
    # Define the output directory.
    output_dir = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/KEGG_Graphs_Generated_13AUG2026"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Read your Excel file containing module expressions.
    input_excel = "/content/drive/MyDrive/Lab Work/Bayesian_Graph_Metabolic_Paper/KEGG_Graph_Generation/KEGG_Prokaryote_Modules_with_Definitions.xlsx"
    data = pd.read_excel(input_excel)
    print(data['Module_Expression'])

    MODULE_EXPRESSION_LOOKUP = load_module_expressions(input_excel)

    # Create new columns for the output file paths if they don't already exist.
    for col in ["Adjacency_File", "Node_List_File", "Graph_PNG_File", "Module_Paths_File"]:
        if col not in data.columns:
            data[col] = ""

    # Define the output Excel file path.
    output_excel = os.path.join(output_dir, "KEGG_Graphs_Generated.xlsx")

    # Process only rows that have blank values in the output columns.
    for index, row in data.iterrows():
        if (pd.isna(row["Adjacency_File"]) or row["Adjacency_File"] == "") and \
           (pd.isna(row["Node_List_File"]) or row["Node_List_File"] == "") and \
           (pd.isna(row["Graph_PNG_File"]) or row["Graph_PNG_File"] == "") and \
           (pd.isna(row["Module_Paths_File"]) or row["Module_Paths_File"] == ""):

            module_id = str(row['Module ID']).strip()  # Use Module ID from the Excel file
            print(f"\nProcessing module with ID {module_id}:")

            raw_expr = row.get("Module_Expression", None)
            if raw_expr is None or (isinstance(raw_expr, float) and pd.isna(raw_expr)) or str(raw_expr).strip() == "":
                print(f"Skipping {module_id}: empty Module_Expression")
                continue


            adj_path, node_list_path, png_path, paths_file_path = main(str(row['Module_Expression']), output_dir, module_id)

            # Write the generated file locations back to the DataFrame.
            data.at[index, "Adjacency_File"] = adj_path
            data.at[index, "Node_List_File"] = node_list_path
            data.at[index, "Graph_PNG_File"] = png_path
            data.at[index, "Module_Paths_File"] = paths_file_path

            # Immediately write the updated DataFrame back to the Excel file.
            data.to_excel(output_excel, index=False)
            print(f"Excel file updated for module {module_id} at index {index}.")

    print(f"Final updated Excel file saved to: {output_excel}")

Streaming output truncated to the last 5000 lines.
Preprocessed Input:
START K22113 K01957 K22114 SINK
Parse Tree:
expr
  top
    step
      chain
        factor
          unit
            start_node
     
    step
      chain
        factor
          unit
            ko_id	K22113
     
    step
      chain
        factor
          unit
            ko_id	K01957
     
    step
      chain
        factor
          unit
            ko_id	K22114
     
    step
      chain
        factor
          unit
            sink_node

AST:
Sequence([Sequence([START, K22113_1, K01957_2, K22114_3, SINK])])

Enumerated Paths:
START -> K22113_1 -> K01957_2 -> K22114_3 -> SINK
Adjacency List:
START: ['K22113_1']
K22113_1: ['K01957_2']
K01957_2: ['K22114_3']
K22114_3: ['SINK']

Node List:
['K01957_2', 'K22113_1', 'K22114_3', 'SINK', 'START']
Node Levels:
K01957_2: Level 3
K22113_1: Level 2
K22114_3: Level 4
SINK: Level 5
START: Level 0
Graph saved as PNG: /content/drive/MyDrive/Lab Work/Bayesian_Graph_Meta